In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
import mlflow
import mlflow.xgboost
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import os

db_path = os.path.join(os.getcwd(), "mlflow.db")
tracking_uri = f"sqlite:///{db_path}"

mlflow.set_tracking_uri(tracking_uri)
mlflow.set_experiment("BEAM")

file_path = r"C:\Users\nilab\Documents\artin\ENB2012_data.csv"
df = pd.read_csv(file_path).dropna()

X = df.drop(columns=['Y1', 'Y2']).values.astype('float32')
y = df[['Y1', 'Y2']].values.astype('float32')

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

depth_options = [3, 6, 9]

print("Starting BEAM Optimization Pipeline...")

for depth in depth_options:
    run_name = f"Pipeline_XGB_Depth_{depth}"
    
    with mlflow.start_run(run_name=run_name):
        params = {
            "objective": "reg:squarederror",
            "n_estimators": 150,
            "learning_rate": 0.05,
            "max_depth": depth,
            "random_state": 42
        }
        
        mlflow.log_params(params)
        
        model = xgb.XGBRegressor(**params)
        model.fit(X_train, y_train)
        
        predictions = model.predict(X_test)
        mse = mean_squared_error(y_test, predictions)
        r2 = r2_score(y_test, predictions)
        
        mlflow.log_metric("mse", mse)
        mlflow.log_metric("r2_score", r2)
        
        mlflow.xgboost.log_model(model, name="model")
        
        print(f"Logged {run_name} | R2: {r2:.4f} | MSE: {mse:.4f}")

print("\n--- All runs complete ---")

Starting BEAM Optimization Pipeline...
Logged Pipeline_XGB_Depth_3 | R2: 0.9832 | MSE: 1.5732
Logged Pipeline_XGB_Depth_6 | R2: 0.9907 | MSE: 0.8719
Logged Pipeline_XGB_Depth_9 | R2: 0.9857 | MSE: 1.3342

--- All runs complete ---
